# Hindi Restoration + Translation — Local Demo

This notebook runs the three assignment stages with the trained LoRA adapters and the local IndicTrans2 copy. Models are loaded one at a time, so it stays well under the 1.5 GB GPU budget.

**Environment:** Python 3.12, torch 2.15 (Apple MPS), transformers 4.44.2, peft 0.13.2. Run time is roughly 2-3 minutes.


In [1]:
from pathlib import Path
import os, sys, json

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print('Project root:', ROOT)
print('Python:', sys.version.split()[0])


Project root: /Users/vaibhavparmar/Documents/nlp-zero-to-hero
Python: 3.12.13


## Stage 1 — punctuation restoration

A `bert-base-multilingual-cased` + LoRA token classifier predicts one label per word. The cell below compares predictions with gold labels from the held-out punctuation test set.


In [2]:
from src.constants import ID2LABEL, LABEL2ID
from src.punctuation.predict import PunctuationRestorer
from src.utils import read_jsonl

row = read_jsonl('data/processed/punctuation/test.jsonl')[0]
restorer = PunctuationRestorer('models/punctuation')
predicted = restorer.predict_labels(row['words'])
truth = [LABEL2ID[label] for label in row['labels']]
accuracy = sum(p == t for p, t in zip(predicted, truth)) / len(truth)
print(f'words in window: {len(truth)}')
for word, true_id, pred_id in list(zip(row['words'], truth, predicted))[:12]:
    print(f'{word:18s} true={ID2LABEL[true_id]:12s} pred={ID2LABEL[pred_id]}')
print(f'window token accuracy: {accuracy:.3f}')
restorer.close()


Some weights of BertForTokenClassification were not initialized from the model checkpoint at bert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


words in window: 73
अपने               true=none         pred=none
तरह                true=none         pred=none
का                 true=none         pred=none
राज्य-प्रबन्धन     true=none         pred=none
विषयक              true=none         pred=none
यह                 true=none         pred=none
प्राचीनतम          true=none         pred=none
ग्रन्थ             true=none         pred=none
है                 true=sentence_end pred=sentence_end
इसमें              true=none         pred=none
राज्यव्यवस्था      true=comma        pred=comma
कृषि               true=comma        pred=comma
window token accuracy: 0.849


## Stage 2 — sentence ordering

The second LoRA adapter scores `P(A comes before B)`. A correct-order pair should score above 0.5 and the reversed pair below 0.5.


In [3]:
from src.ordering.decode import OrderScorer
from src.utils import read_jsonl

pair = read_jsonl('data/processed/ordering/test.jsonl')[0]
scorer = OrderScorer('models/ordering')
matrix = scorer.score_matrix([pair['sentence_a'], pair['sentence_b']])
print('gold label:', pair['label'], '(1 = A before B)')
print('P(A before B):', round(float(matrix[0, 1]), 3))
print('P(B before A):', round(float(matrix[1, 0]), 3))
print('A:', pair['sentence_a'][:120])
print('B:', pair['sentence_b'][:120])
scorer.close()


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


gold label: 1 (1 = A before B)
P(A before B): 0.711
P(B before A): 0.289
A: यह बाद में ऑगस्ट कॉम्ट(1798-1857 द्वारा 1838 में स्थापित किया गया
B: इससे पहले कॉम्ट ने सामाजिक भौतिकी शब्द का इस्तेमाल किया था लेकिन बाद में वह दूसरों द्वारा अपनाया गया विशेष रूप से बेल्जि


## Stage 3 — Hindi to English translation

`IndicTrans2` (200M) with `IndicTransToolkit` preprocessing. No LLM and no fallback translator are used anywhere.


In [4]:
from src.translation.indictrans import IndicTrans2Translator

translator = IndicTrans2Translator('models/translation/indictrans2-indic-en-200M')
sentences = [
    'भारत एक विशाल देश है।',
    'यह एक परीक्षण है।',
    'कृपया अपनी रिपोर्ट कल तक जमा करें।',
]
for hi, en in zip(sentences, translator.translate(sentences)):
    print(f'HI: {hi}')
    print(f'EN: {en}\n')
translator.close()


HI: भारत एक विशाल देश है।
EN: India is a huge country.

HI: यह एक परीक्षण है।
EN: It's a test.

HI: कृपया अपनी रिपोर्ट कल तक जमा करें।
EN: Please submit your report by tomorrow.



## End-to-end artifacts

The pipeline has already written all seven samples. This cell checks that Hindi/English sentences and groups stay aligned.


In [5]:
summary = json.loads(Path('reports/run_summary.json').read_text())
print('timings (seconds):', summary['timings_seconds'])
for stem, info in summary['files'].items():
    hi = Path(f'outputs/{stem}.sentences_hi.txt').read_text().splitlines()
    en = Path(f'outputs/{stem}.sentences_en.txt').read_text().splitlines()
    gh = Path(f'outputs/{stem}.groups_hi.txt').read_text().splitlines()
    ge = Path(f'outputs/{stem}.groups_en.txt').read_text().splitlines()
    ok = len(hi) == len(en) == info['sentences'] and len(gh) == len(ge) == info['groups']
    print(f"{stem}: {info['sentences']} sentences, {info['groups']} groups, aligned={ok}")
print('\nFirst 3 ordered Hindi sentences:')
for line in Path('outputs/sample-1.sentences_hi.txt').read_text().splitlines()[:3]:
    print('-', line)
print('\nFirst 3 English translations:')
for line in Path('outputs/sample-1.sentences_en.txt').read_text().splitlines()[:3]:
    print('-', line)


timings (seconds): {'punctuation': 2.31, 'ordering': 130.53, 'translation': 102.17}
sample-1: 52 sentences, 11 groups, aligned=True
sample-2: 55 sentences, 11 groups, aligned=True
sample-3: 49 sentences, 10 groups, aligned=True
sample-4: 53 sentences, 11 groups, aligned=True
sample-5: 55 sentences, 11 groups, aligned=True
sample-6: 80 sentences, 16 groups, aligned=True
sample-7: 99 sentences, 20 groups, aligned=True

First 3 ordered Hindi sentences:
- तब, महालक्ष्मी हाथ में कमल धारण किए हुए, उनके सामने प्रकट हुई थीं।
- इसका नाम बनारस या वाराणसी भी है।
- वे इस सोच में पड़ गए, कि युद्ध में मारे गए, असंख्य, कुटुम्बियों, बंधुओं, व निर्दोष, लोगों की आत्मा की शांति के लिए क्या उपाय या संस्कार किया जाए।

First 3 English translations:
- Then, Mahalakshmi appeared before him, holding a lotus in her hand.
- It is also called Banaras or Varanasi.
- They wondered what measures or rites should be taken for the peace of the souls of the countless, relatives, brothers, and innocent people who died in

## Evaluation report

Scores from `reports/evaluation.json` (subset adapters; the full-data Colab recipe should improve the ordering stage).


In [6]:
ev = json.loads(Path('reports/evaluation.json').read_text())
print('punctuation accuracy / boundary F1:', ev['punctuation']['accuracy'], '/', ev['punctuation']['boundary']['f1'])
print('ordering pair accuracy:', ev['ordering']['pairs']['accuracy'])
print('ordering Borda Kendall tau:', ev['ordering']['kendall_tau']['borda']['mean_tau'])
print('translation BLEU / chrF:', ev['translation']['bleu'], '/', ev['translation']['chrf'])


punctuation accuracy / boundary F1: 0.8724 / 0.8842
ordering pair accuracy: 0.6543
ordering Borda Kendall tau: 0.2007
translation BLEU / chrF: 34.125 / 59.686


## Reproduce from the command line

```bash
python -m src.pipeline \
  --translation-model models/translation/indictrans2-indic-en-200M \
  --translation-batch-size 8

python -m src.evaluate \
  --translation-model models/translation/indictrans2-indic-en-200M \
  --max-translation-examples 100
```
